# Laboratorio II – Imagen del dinosaurio (text-to-image en Colab)

**Entorno de ejecución → Cambiar tipo → GPU T4.**
Parte A: genera la imagen del dinosaurio elegido (evidencia del paso 4 de la guía).
Parte B: deja el modelo como servicio HTTP (`POST /generar`) expuesto con ngrok para la app web.

Credenciales: guarda `NGROK_AUTHTOKEN` (y opcionalmente `NGROK_DOMAIN`) en **Secretos de Colab** (icono de llave), nunca en el código.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U diffusers transformers accelerate safetensors fastapi uvicorn pyngrok

In [ ]:
import torch, io, base64, json, threading
from diffusers import AmusedPipeline

MODELO_IMAGEN = "amused/amused-512"
pipe = AmusedPipeline.from_pretrained(MODELO_IMAGEN, variant="fp16", torch_dtype=torch.float16).to("cuda")
pipe.vqvae.to(torch.float32)   # el VQ-VAE puede producir NaN en fp16
print("Modelo cargado:", MODELO_IMAGEN)

## Parte A – Imagen del dinosaurio elegido
Pega la identidad generada por Ollama (paso 2). El prompt se arma con el **nombre** y los **rasgos visuales**.

In [ ]:
identidad = {
    "nombre": "Mangosaurus",
    "prompt_imagen": "a large bipedal theropod dinosaur with a round mango-colored body, red and yellow gradient skin, small arms, long tail, tropical forest background",
}

NEGATIVO = "blurry, low quality, text, watermark, cartoon, extra limbs, deformed"

def construir_prompt(identidad):
    base = identidad.get("prompt_imagen") or identidad.get("apariencia", "")
    return f"{identidad['nombre']} dinosaur, {base}, full body, natural history illustration, detailed"

def generar_imagen(prompt, semilla=0, pasos=12, negativo=NEGATIVO):
    g = torch.Generator("cuda").manual_seed(semilla)
    return pipe(prompt, negative_prompt=negativo, num_inference_steps=pasos, generator=g).images[0]

prompt = construir_prompt(identidad)
print(prompt)
imagen = generar_imagen(prompt, semilla=0)
imagen.save(f"{identidad['nombre'].lower()}.png")
imagen

Si la imagen no coincide con los rasgos, cambia la `semilla` o simplifica el prompt (menos adjetivos, más concretos).
Descarga el PNG y guárdalo en `evidencias/` del repositorio junto con el prompt usado.

## Parte B – Servicio HTTP para la app
Contrato:
- `GET /salud` → `{"ok": true, "modelo": "..."}`
- `POST /generar` con `{"prompt": str, "semilla": int?, "pasos": int?}` → `{"image_base64": str, "mime": "image/png", "modelo": str, "prompt": str}`

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

app = FastAPI(title="Servicio de imagen de dinosaurios")
candado = threading.Lock()   # una generación a la vez en la GPU

class Peticion(BaseModel):
    prompt: str
    semilla: int = 0
    pasos: int = 12

@app.get("/salud")
def salud():
    return {"ok": True, "modelo": MODELO_IMAGEN}

@app.post("/generar")
def generar(p: Peticion):
    with candado:
        img = generar_imagen(p.prompt[:400], p.semilla, max(4, min(p.pasos, 30)))
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return {"image_base64": base64.b64encode(buf.getvalue()).decode(), "mime": "image/png",
            "modelo": MODELO_IMAGEN, "prompt": p.prompt}

servidor = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=servidor.run, daemon=True).start()

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
try:
    dominio = userdata.get("NGROK_DOMAIN")
except Exception:
    dominio = None
tunel = ngrok.connect(8000, "http", domain=dominio) if dominio else ngrok.connect(8000, "http")
print("IMAGEN_URL =", tunel.public_url)

In [ ]:
# Prueba del servicio a través del túnel (lo mismo que hará la app)
import requests
from PIL import Image
r = requests.post(f"{tunel.public_url}/generar", json={"prompt": prompt, "semilla": 1},
                  headers={"ngrok-skip-browser-warning": "1"}, timeout=120).json()
Image.open(io.BytesIO(base64.b64decode(r["image_base64"])))

Deja esta pestaña abierta mientras usas la app: si Colab se desconecta, vuelve a ejecutar todo el notebook.